# Radio Signal Parameter Estimation: Issues & Solutions Log

This notebook documents the process of training a physics-informed neural network to estimate the phase offset of a 16-QAM radio signal and reconstruct it. Below is a summary of the critical issues encountered during development and the solutions that led to a successful model (BER < 1%).

### 1. The "Symmetry Ambiguity" Problem

* **Issue:** The initial model converged to a loss of ~1.0 (random guessing) and a BER of ~0.85.
* **Root Cause:** 16-QAM is rotationally symmetric every $90^\circ$ ($\pi/2$). Without an external reference, the network cannot distinguish between true $\theta$ and $\theta+90^\circ$. It attempts to average these possibilities, leading to a destructive mean of 0.
* **Solution:** **Pilot-Aided Estimation**. We explicitly feed the first $N=4$ known symbols ("pilots") into the network. These act as a "phase anchor," breaking the symmetry and allowing the network to lock onto the correct quadrant.

### 2. The "Lazy Network" / Vanishing Gradient Problem

* **Issue:** Even with pilots, the network initially ignored them because the sparse pilot signal ($8$ floats) was drowned out by the noisy data signal ($1024$ floats).
* **Root Cause:** The network struggled to learn the specific complex-conjugate arithmetic required to extract phase from pilots from scratch (the optimization landscape was too flat).
* **Solution:** **Physics-Informed Residual Learning**. We calculate a "Classical Hint" (a rough Coarse Estimate using the pilots via Least Squares) and feed this vector $[\cos \hat{\theta}, \sin \hat{\theta}]$ into the network.
* *Result:* The NN no longer needs to learn the phase from zero; it only needs to learn the **Residual Correction** ($\Delta \theta$) to account for noise and non-linearities, which is a much easier task.



### 3. Input Normalization & SNR Curriculum

* **Issue:** Gradients were unstable, and the MLP struggled to converge on raw IQ data.
* **Solution:**
1. **Batch Normalization:** Added `BatchNorm1d` immediately after flattening inputs to center the data and keep variance unit-scale.
2. **SNR Randomization:** We trained on a dynamic SNR range ($15 - 30$ dB). This prevented the model from overfitting to clean signals and forced it to learn robust feature extraction for noisy environments.



### 4. Manifold-Aware Loss

* **Issue:** Using MSE (Mean Squared Error) on raw angles fails because $-\pi$ and $+\pi$ are far apart in Euclidean numbers but identical in physical phase (the "wrap-around" problem).
* **Solution:** We predict a vector $[\cos \theta, \sin \theta]$ and maximize the cosine similarity. The loss function is $L = 1 - \cos(\theta_{pred} - \theta_{true})$, which is differentiable and correctly respects the circular geometry of the phase manifold.

---

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Subset
import matplotlib.pyplot as plt
import random
import time
import torch.nn.functional as F

# -------------------------------------------------------------------------------------
# 1. Configuration
# -------------------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
SEQ_LEN = 256
N_PILOTS = 4
DATA_PATH = "./data/radio_divina_commedia.pth"

# OPTIMIZED SCHEDULE
EPOCHS_ESTIMATOR = 20  
EPOCHS_ORACLE    = 5  
EPOCHS_HANDOVER  = 10  
EPOCHS_ROBUST    = 5  

class Colors:
    GREEN = '\033[92m'; RED = '\033[91m'; RESET = '\033[0m'; BOLD = '\033[1m'; YELLOW = '\033[93m'; CYAN = '\033[96m'
    BG_GREEN = '\033[42m'; FG_WHITE = '\033[97m'

print(f"Using device: {DEVICE}")

# -------------------------------------------------------------------------------------
# 2. Helpers
# -------------------------------------------------------------------------------------
def build_verified_constellation(path):
    if not os.path.exists(path):
        b = torch.tensor([-3.,-1.,1.,3.], device=DEVICE)
        g = torch.meshgrid(b, b, indexing='ij')
        return ((g[0] + 1j*g[1]).flatten() / np.sqrt(10)).to(DEVICE)
    ckpt = torch.load(path, map_location=DEVICE)
    iq_c = ckpt['iq_clean']
    z_c = torch.complex(iq_c[0,0,:], iq_c[0,1,:])
    z_np = z_c.cpu().numpy()
    const_np = np.unique(z_np)
    if len(const_np) < 16: 
        b = torch.tensor([-3.,-1.,1.,3.], device=DEVICE)
        g = torch.meshgrid(b, b, indexing='ij')
        const = (g[0] + 1j*g[1]).flatten()
        const = const / torch.sqrt(torch.mean(torch.abs(const)**2))
        return const.to(DEVICE)
    return torch.from_numpy(const_np).to(DEVICE)

CONST_TENSOR = build_verified_constellation(DATA_PATH)

def classical_pilot_estimate(noisy_pilots, clean_pilots):
    phasor = (noisy_pilots * torch.conj(clean_pilots)).sum(dim=1)
    norm = torch.abs(phasor) + 1e-8
    return torch.stack([phasor.real/norm, phasor.imag/norm], dim=1).float()

def demap_16qam(iq_tensor):
    dist = torch.abs(iq_tensor.unsqueeze(-1) - CONST_TENSOR.view(1, 1, -1))
    return torch.argmin(dist, dim=-1)

def calc_ber(pred_idx, true_idx):
    pred_idx = pred_idx.long()
    true_idx = true_idx.long()
    diff = pred_idx ^ true_idx
    errors = 0
    for i in range(4):
        errors += ((diff >> i) & 1).sum()
    return errors.float() / (pred_idx.numel() * 4)

def decode_text(indices, mask):
    try:
        ints = indices.cpu().numpy().astype(np.uint8)
        bits = np.unpackbits(ints[:, None], axis=1)[:, -4:].flatten()
        m = mask.cpu().numpy().flatten()[:len(bits)]
        clean = np.bitwise_xor(bits, m)
        return np.packbits(clean).tobytes().replace(b'\x00', b'').decode('utf-8', 'ignore')
    except Exception as e: 
        return f"[Error: {e}]"

# -------------------------------------------------------------------------------------
# 3. Dataset
# -------------------------------------------------------------------------------------
def load_and_split_data(path, batch_size=BATCH_SIZE, seed=42):
    global SEQ_LEN
    print(f"Loading {path}...")
    ckpt = torch.load(path, map_location=DEVICE)
    x = ckpt['iq_noisy'].float()
    target_iq = ckpt['iq_clean'].float()
    
    y_phi = ckpt['phase_labels'].float().reshape(-1)
    y_cfo = ckpt['cfo_labels'].float().reshape(-1)
    y_snr = ckpt.get('snr_db', torch.zeros_like(y_phi)).float().reshape(-1)
    bits = ckpt['bits'] 
    
    if 'scramble_mask' in ckpt:
        masks = ckpt['scramble_mask']
    else:
        print(f"{Colors.RED}WARNING: 'scramble_mask' missing.{Colors.RESET}")
        masks = torch.zeros_like(bits) 

    if x.shape[2] != SEQ_LEN: SEQ_LEN = x.shape[2]
    
    b_re = bits.reshape(bits.shape[0], -1, 4).long()
    z_idx = (b_re[:,:,0]*8 + b_re[:,:,1]*4 + b_re[:,:,2]*2 + b_re[:,:,3]*1)
    
    target_bits = bits.reshape(bits.shape[0], SEQ_LEN, 4).permute(0, 2, 1).float()
    
    ds = TensorDataset(x, target_iq, y_phi, y_cfo, y_snr, target_bits, z_idx, masks)
    idx = list(range(len(ds)))
    random.Random(seed).shuffle(idx)
    
    train_ds = Subset(ds, idx[:int(0.80*len(ds))])
    val_ds   = Subset(ds, idx[int(0.80*len(ds)):int(0.90*len(ds))])
    test_ds  = Subset(ds, idx[int(0.90*len(ds)):])
    
    return (DataLoader(train_ds, batch_size, shuffle=True), 
            DataLoader(val_ds, batch_size, False), 
            DataLoader(test_ds, batch_size, False))

train_loader, val_loader, test_loader = load_and_split_data(DATA_PATH)

# -------------------------------------------------------------------------------------
# 4. Architecture
# -------------------------------------------------------------------------------------
class LinearAttention(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.heads = heads
        self.dim_head = dim // heads
        self.scale = self.dim_head ** -0.5
        self.to_qkv = nn.Linear(dim, dim * 3, bias=False)
        self.to_out = nn.Linear(dim, dim)
        self.gamma = nn.Linear(dim, dim)
        self.beta = nn.Linear(dim, dim)

    def forward(self, x, latent_z):
        style_scale = self.gamma(latent_z).unsqueeze(1)
        style_shift = self.beta(latent_z).unsqueeze(1)
        x = x * (1 + style_scale) + style_shift
        b, n, d = x.shape
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: t.view(b, n, self.heads, self.dim_head).transpose(1, 2), qkv)
        q = q.softmax(dim=-1) * self.scale
        k = k.softmax(dim=-2)
        context = torch.matmul(k.transpose(-1, -2), v) 
        out = torch.matmul(q, context)
        out = out.transpose(1, 2).reshape(b, n, d)
        return self.to_out(out)

class ResidualBlock1D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(x + self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))))

class HybridNeuralReceiver(nn.Module):
    def __init__(self, seq_len=SEQ_LEN, n_pilots=N_PILOTS):
        super().__init__()
        in_dim = (2 * seq_len) + (2 * n_pilots) + 2
        self.input_norm = nn.BatchNorm1d(in_dim)
        self.estimator = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(), nn.BatchNorm1d(512),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256),
            nn.Linear(256, 3) 
        )
        self.latent_proj = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.Tanh())
        self.feat_extract = nn.Conv1d(2, 64, 7, padding=3)
        self.res1 = ResidualBlock1D(64)
        self.attn = LinearAttention(64) 
        self.norm_attn = nn.LayerNorm(64)
        self.res2 = ResidualBlock1D(64)
        self.final_conv = nn.Conv1d(64, 32, 1)
        self.bit_head = nn.Conv1d(32, 4, 1)

    def forward(self, x, pilots, hint, gt_phi=None, gt_cfo=None, mix_ratio=0.0):
        x_flat = x.reshape(x.size(0), -1)
        p_flat = pilots.reshape(pilots.size(0), -1)
        est_feats = self.input_norm(torch.cat([x_flat, p_flat, hint], dim=1))
        est_params = self.estimator(est_feats)
        est_phi = torch.atan2(est_params[:, 1], est_params[:, 0])
        est_cfo = est_params[:, 2] / 100.0
        
        if gt_phi is not None and mix_ratio < 1.0:
            phi = (1 - mix_ratio) * gt_phi + mix_ratio * est_phi
            cfo = (1 - mix_ratio) * gt_cfo + mix_ratio * est_cfo
        else:
            phi, cfo = est_phi, est_cfo
            
        z_input = torch.stack([torch.cos(phi), torch.sin(phi), cfo * 100.0], dim=1)
        z = self.latent_proj(z_input)

        B, _, L = x.shape
        t = torch.arange(L, device=x.device).float().unsqueeze(0)
        phase_ramp = phi.unsqueeze(1) + (cfo.unsqueeze(1) * t)
        cos_t, sin_t = torch.cos(-phase_ramp), torch.sin(-phase_ramp)
        r_I = x[:, 0, :] * cos_t - x[:, 1, :] * sin_t
        r_Q = x[:, 0, :] * sin_t + x[:, 1, :] * cos_t
        x_derot = torch.stack([r_I, r_Q], dim=1)
        
        feat = F.relu(self.feat_extract(x_derot))
        feat = self.res1(feat)
        feat_attn = feat.permute(0, 2, 1) 
        feat_attn = self.attn(self.norm_attn(feat_attn), z)
        feat = feat + feat_attn.permute(0, 2, 1) 
        feat = self.res2(feat)
        logits = self.bit_head(F.relu(self.final_conv(feat)))
        return logits, est_phi, est_cfo

# -------------------------------------------------------------------------------------
# 5. Training
# -------------------------------------------------------------------------------------
model = HybridNeuralReceiver(SEQ_LEN).to(DEVICE)
optimizer_est = optim.Adam(model.estimator.parameters(), lr=0.001)
optimizer_dec = optim.Adam(list(model.feat_extract.parameters()) + 
                           list(model.res1.parameters()) + 
                           list(model.attn.parameters()) + 
                           list(model.res2.parameters()) +
                           list(model.final_conv.parameters()) +
                           list(model.bit_head.parameters()) +
                           list(model.latent_proj.parameters()), lr=0.0005)
criterion_mse = nn.MSELoss()
criterion_bce = nn.BCEWithLogitsLoss()

print(f"--- Starting Hybrid Training Schedule ---")
TOTAL_EPOCHS = EPOCHS_ESTIMATOR + EPOCHS_ORACLE + EPOCHS_HANDOVER + EPOCHS_ROBUST

for epoch in range(TOTAL_EPOCHS):
    model.train()
    if epoch < EPOCHS_ESTIMATOR:
        stage = "ESTIMATOR"; mix = 0.0; train_est, train_dec = True, False
    elif epoch < (EPOCHS_ESTIMATOR + EPOCHS_ORACLE):
        stage = "ORACLE"; mix = 0.0; train_est, train_dec = False, True
    elif epoch < (EPOCHS_ESTIMATOR + EPOCHS_ORACLE + EPOCHS_HANDOVER):
        stage = "HANDOVER"
        steps = epoch - (EPOCHS_ESTIMATOR + EPOCHS_ORACLE)
        mix = min(1.0, steps / (EPOCHS_HANDOVER * 0.8))
        train_est, train_dec = False, True 
    else:
        stage = "ROBUSTNESS"; mix = 1.0; train_est, train_dec = False, True
        
    metrics = {'loss':0, 'phi':0, 'cfo_mae':0, 'ber':0}
    
    for batch in train_loader:
        x, _, y_p, y_c, y_s, target_bits, z_idx, _ = batch
        x, y_p, y_c = x.to(DEVICE), y_p.to(DEVICE), y_c.to(DEVICE)
        target_bits = target_bits.to(DEVICE); z_idx = z_idx.to(DEVICE)
        p_ref = CONST_TENSOR[z_idx[:, :N_PILOTS]]
        hint = classical_pilot_estimate(torch.complex(x[:,0], x[:,1])[:, :N_PILOTS], p_ref)
        p_feat = torch.stack([p_ref.real, p_ref.imag], dim=1).float()
        
        y_p_in, y_c_in = y_p, y_c
        if stage == "ROBUSTNESS":
            y_p_in = y_p + (torch.rand_like(y_p) - 0.5) * 0.1 
        
        bit_logits, est_phi, est_cfo = model(x, p_feat, hint, y_p_in, y_c_in, mix_ratio=mix)
        
        loss = 0
        if train_est:
            loss += (1.0 - torch.cos(est_phi - y_p).mean()) + criterion_mse(est_cfo*100, y_c*100)
        if train_dec:
            loss += criterion_bce(bit_logits, target_bits)
            
        if train_est: optimizer_est.zero_grad(); loss.backward(); optimizer_est.step()
        if train_dec: optimizer_dec.zero_grad(); loss.backward(); optimizer_dec.step()
        
        metrics['loss'] += loss.item()
        metrics['phi'] += torch.abs(est_phi - y_p).mean().item()
        metrics['cfo_mae'] += torch.abs(est_cfo - y_c).mean().item()
        with torch.no_grad():
            pred_bits = (bit_logits > 0).float()
            metrics['ber'] += (pred_bits != target_bits).float().mean().item()

    n = len(train_loader)
    print(f"Ep {epoch+1:02d}/{TOTAL_EPOCHS} | {Colors.CYAN}{stage:<10}{Colors.RESET} | Mix: {mix:.2f} | "
          f"Loss: {metrics['loss']/n:.4f} | "
          f"Phi: {metrics['phi']/n:.3f} | CFO: {metrics['cfo_mae']/n:.5f} | "
          f"BER: {metrics['ber']/n*100:.2f}%")

# -------------------------------------------------------------------------------------
# 6. High-Fidelity Benchmark (Clean Visualization)
# -------------------------------------------------------------------------------------
print(f"\n{Colors.BOLD}{'='*120}{Colors.RESET}")
print(f"{Colors.BOLD} FINAL PERFORMANCE REPORT (10 Samples){Colors.RESET}")
print(f"{Colors.BOLD}{'='*120}{Colors.RESET}")

model.eval()
res = {'c_ber':[], 'n_ber':[]}
cnt = 0
SAMPLES_TO_SHOW = 10

# Special aligned decoder for visualization
def decode_text_aligned(indices, mask):
    try:
        ints = indices.cpu().numpy().astype(np.uint8)
        bits = np.unpackbits(ints[:, None], axis=1)[:, -4:].flatten()
        m = mask.cpu().numpy().flatten()[:len(bits)]
        clean = np.bitwise_xor(bits, m)
        # Use errors='replace' to keep byte count consistent, then map unprintables to '.'
        text = np.packbits(clean).tobytes().decode('utf-8', 'replace')
        # Force fixed width (replace newlines/controls with .)
        clean_text = "".join([c if c.isprintable() else '.' for c in text])
        return clean_text
    except: return "." * (len(indices) * 4 // 8)

def color_diff_text(truth, pred):
    out = ""
    length = min(len(truth), len(pred))
    for i in range(length):
        t_char = truth[i]
        p_char = pred[i]
        if t_char == p_char:
            out += f"{Colors.GREEN}{p_char}{Colors.RESET}"
        else:
            out += f"{Colors.RED}{p_char}{Colors.RESET}"
    return out

with torch.no_grad():
    for x, _, y_p, y_c, _, target_bits, z_idx, m in test_loader:
        x, z_idx, m = x.to(DEVICE), z_idx.to(DEVICE), m.to(DEVICE)
        y_p, y_c = y_p.to(DEVICE), y_c.to(DEVICE)
        x_c = torch.complex(x[:,0], x[:,1])
        
        # --- 1. Classical ---
        p_ref = CONST_TENSOR[z_idx[:, :N_PILOTS]]
        theta = torch.angle((x_c[:, :N_PILOTS] * torch.conj(p_ref)).sum(dim=1))
        rx_cl = x_c * torch.exp(-1j * theta.unsqueeze(1)) 
        idx_cl = demap_16qam(rx_cl)
        
        # --- 2. Neural ---
        p_feat = torch.stack([p_ref.real, p_ref.imag], dim=1).float()
        hint = classical_pilot_estimate(x_c[:, :N_PILOTS], p_ref)
        bit_logits, est_phi, est_cfo = model(x, p_feat, hint, mix_ratio=1.0)
        
        pred_bits = (bit_logits > 0).long()
        idx_nn = (pred_bits[:,0,:]*8 + pred_bits[:,1,:]*4 + pred_bits[:,2,:]*2 + pred_bits[:,3,:]*1)
        
        for i in range(x.size(0)):
            ber_cl = calc_ber(idx_cl[i], z_idx[i]).item()
            ber_nn = calc_ber(idx_nn[i], z_idx[i]).item()
            res['c_ber'].append(ber_cl)
            res['n_ber'].append(ber_nn)
            
            if cnt < SAMPLES_TO_SHOW:
                phi_err = est_phi[i].item() - y_p[i].item()
                phi_err = (phi_err + np.pi) % (2 * np.pi) - np.pi
                
                pay = slice(N_PILOTS, None)
                truth = decode_text_aligned(z_idx[i, pay], m[i, N_PILOTS*4:])
                txt_c = decode_text_aligned(idx_cl[i, pay], m[i, N_PILOTS*4:])
                txt_n = decode_text_aligned(idx_nn[i, pay], m[i, N_PILOTS*4:])
                
                # --- FORMATTING ---
                print(f"{Colors.BOLD}SAMPLE {cnt+1:02d}{Colors.RESET} | Est Phase Err: {abs(phi_err):.4f} rad")
                
                # Truncate all to length of truth to enforce alignment
                L = len(truth)
                t_str = truth[:L]
                c_str = txt_c[:L]
                n_str = txt_n[:L]
                
                print(f" {Colors.CYAN}GT    :{Colors.RESET} {t_str}")
                
                diff_c = color_diff_text(t_str, c_str)
                print(f" {Colors.YELLOW}CLASS :{Colors.RESET} {diff_c} {Colors.BOLD}[BER: {ber_cl*100:.1f}%]{Colors.RESET}")
                
                diff_n = color_diff_text(t_str, n_str)
                if ber_nn < 0.01:
                    prefix = f"\033[42m\033[97m NEURAL \033[0m" 
                    print(f"{prefix} {diff_n} {Colors.BOLD}[BER: {ber_nn*100:.2f}%]{Colors.RESET}")
                else:
                    prefix = f" {Colors.BOLD}NEURAL:{Colors.RESET}"
                    print(f"{prefix} {diff_n} {Colors.BOLD}[BER: {ber_nn*100:.1f}%]{Colors.RESET}")

                print(f"{Colors.BOLD}{'-'*120}{Colors.RESET}")
                cnt += 1

print("\n" + "="*40)
print("AGGREGATE STATISTICS")
print("="*40)
print(f"Classical Mean BER: {np.mean(res['c_ber'])*100:.2f}%")
print(f"Neural Mean BER:    {Colors.BOLD}{np.mean(res['n_ber'])*100:.2f}%{Colors.RESET}")

Using device: cpu
Loading ./data/radio_divina_commedia.pth...
--- Starting Hybrid Training Schedule ---
Ep 01/40 | ESTIMATOR  | Mix: 0.00 | Loss: 2.4391 | Phi: 1.743 | CFO: 0.01081 | BER: 49.31%
Ep 02/40 | ESTIMATOR  | Mix: 0.00 | Loss: 1.1761 | Phi: 1.281 | CFO: 0.00705 | BER: 49.32%
Ep 03/40 | ESTIMATOR  | Mix: 0.00 | Loss: 0.5904 | Phi: 1.006 | CFO: 0.00433 | BER: 49.31%
Ep 04/40 | ESTIMATOR  | Mix: 0.00 | Loss: 0.3947 | Phi: 0.840 | CFO: 0.00340 | BER: 49.32%
Ep 05/40 | ESTIMATOR  | Mix: 0.00 | Loss: 0.3182 | Phi: 0.708 | CFO: 0.00323 | BER: 49.33%
Ep 06/40 | ESTIMATOR  | Mix: 0.00 | Loss: 0.2464 | Phi: 0.599 | CFO: 0.00293 | BER: 49.31%
Ep 07/40 | ESTIMATOR  | Mix: 0.00 | Loss: 0.2099 | Phi: 0.529 | CFO: 0.00288 | BER: 49.32%
Ep 08/40 | ESTIMATOR  | Mix: 0.00 | Loss: 0.1874 | Phi: 0.462 | CFO: 0.00280 | BER: 49.32%
Ep 09/40 | ESTIMATOR  | Mix: 0.00 | Loss: 0.1619 | Phi: 0.417 | CFO: 0.00266 | BER: 49.31%
Ep 10/40 | ESTIMATOR  | Mix: 0.00 | Loss: 0.1528 | Phi: 0.357 | CFO: 0.00263 

# Parameter estimation to be improved